In [ ]:
class CashFlows:
    def __init__(self):
        self.maturities = []  # An attribute to store when cash flows occur
        self.amounts = []     # An attribute to store the cash flow amounts
    
    def add_cash_flow(self, maturity, amount):
        """Add a cash flow to the cash flow list."""
        self.maturities.append(maturity)
        self.amounts.append(amount)
    
    def get_cash_flow(self, maturity):
        """Get the cash flow amount for a specific maturity."""
        if maturity in self.maturities:
            return self.amounts[self.maturities.index(maturity)]
        else:
            return None
    
    def get_maturities(self):
        """Return a list of all maturities."""
        return list(self.maturities)
    
    def get_amounts(self):
        """Return a list of all cash flow amounts."""
        return list(self.amounts)
    
    def get_cash_flows(self):
        """Return all cash flows as (maturity, amount) pairs."""
        return list(zip(self.maturities, self.amounts))

    a_cashflows_object = CashFlows()
    
    print('a cashflow object')

    a_cashflows_object.add_cash_flow(1,100)
    a_cashflows_object.add_cash_flow(2,150)
    a_cashflows_object.add_cash_flow(1.5,250)

    print(a_cashflows_object.get_cash_flows())
    print(a_cashflows_object.get_cash_flow(1.5))
    print(a_cashflows_object.get_maturities())

    another_cashflows_object = CashFlows()

    print('\n another cashflow object')



a cashflow object
[(1, 100), (2, 150), (1.5, 250)]
250
[1, 2, 1.5]

 another cashflow object


NameError: name 'math' is not defined

In [29]:

import math
import numpy as np
import importlib
import pandas as pd
import tabulate as tb

class ZeroCurve:
    def __init__(self):
        # set up empty lists to store the curve data
        self.maturities = []
        self.zero_rates = []
        self.AtMats = []
        self.discount_factors = []
    
    def add_zero_rate(self, maturity, zero_rate):
        """Add a zero rate to the curve and calculate corresponding discount factor and AtMat"""
        self.maturities.append(maturity)
        self.zero_rates.append(zero_rate)
        self.AtMats.append(math.exp(zero_rate*maturity))
        self.discount_factors.append(1/self.AtMats[-1])

    def add_discount_factor(self, maturity, discount_factor):
        """Add a discount factor to the curve and calculate corresponding zero rate and AtMat"""
        self.maturities.append(maturity)
        self.discount_factors.append(discount_factor)
        self.AtMats.append(1/discount_factor)
        self.zero_rates.append(math.log(1/discount_factor)/maturity)
    
    def get_AtMat(self, maturity):
        """Get the amount at maturity for a given time point (with interpolation if needed)"""
        if maturity in self.maturities:
            return self.AtMats[self.maturities.index(maturity)]
        else:
            return exp_interp(self.maturities, self.AtMats, maturity)

    def get_discount_factor(self, maturity):
        """Get the discount factor for a given maturity (with interpolation if needed)"""
        if maturity in self.maturities:
            return self.discount_factors[self.maturities.index(maturity)]
        else:
            return exp_interp(self.maturities, self.discount_factors, maturity)

    def get_zero_rate(self, maturity):
        """Get the zero rate for a given maturity (with interpolation if needed)"""
        if maturity in self.maturities:
            return self.zero_rates[self.maturities.index(maturity)]
        else:
            return math.log(self.get_AtMat(maturity))/maturity
        
    def get_zero_curve(self):
        """Return the complete zero curve as maturities and discount factors"""
        return self.maturities, self.discount_factors
    
    def npv(self, cash_flows):
        """Calculate the net present value of a cash flow stream"""
        npv = 0
        for maturity in cash_flows.get_maturities():
            npv += cash_flows.get_cash_flow(maturity)*self.get_discount_factor(maturity)
        return npv
        

def exp_interp(xs, ys, x):
    """
    Interpolates a single point for a given value of x 
    using continuously compounded rates.

    Parameters:
    xs (list or np.array): Vector of x values sorted by x.
    ys (list or np.array): Vector of y values.
    x (float): The x value to interpolate.

    Returns:
    float: Interpolated y value.
    """
    xs = np.array(xs)
    ys = np.array(ys)
    
    # Find the interval [x0, x1] where x0 <= x <= x1
    idx = np.searchsorted(xs, x) - 1
    x0, x1 = xs[idx], xs[idx + 1]
    y0, y1 = ys[idx], ys[idx + 1]
    
    # Calculate the continuously compounded rate
    rate = (np.log(y1) - np.log(y0)) / (x1 - x0)
    
    # Interpolate the y value for the given x
    y = y0 * np.exp(rate * (x - x0))
    
    return y


# Create an instance of the ZeroCurve class
# This initializes an empty curve ready to accept rate data
zc = ZeroCurve()

# Add zero rates to the curve
# These rates represent annual interest rates for zero-coupon bonds
zc.add_zero_rate(0.5, 0.0125)  # 6-month rate
zc.add_zero_rate(1, 0.015)  # 1.5% for 1 year
zc.add_zero_rate(2, 0.025)  # 2.5% for 2 years
zc.add_zero_rate(3, 0.035)  # 3.5% for 3 years
zc.add_zero_rate(4, 0.045)  # 4.5% for 4 years

# Demonstrate retrieving zero rates
print("2.5-year zero rate:", zc.get_zero_rate(2.5))

# Calculate and display discount factors
# Discount factors represent the present value of 1 unit of currency
print("1-year discount factor:", zc.get_discount_factor(1))
print("2-year discount factor:", zc.get_discount_factor(2))
print("3-year discount factor:", zc.get_discount_factor(3))
print("4-year discount factor:", zc.get_discount_factor(4))

# Demonstrate interpolation for a non-standard maturity
maturity_lookup = 1.5
print(f"Zero rate for {maturity_lookup} years:", zc.get_zero_rate(maturity_lookup))
print(f"Amount at Maturity for {maturity_lookup} years:", zc.get_AtMat(maturity_lookup))
print(f"Discount factor for {maturity_lookup} years:", zc.get_discount_factor(maturity_lookup))

# Get the complete zero curve data
print("Complete zero curve:", zc.get_zero_curve())

# Create a pandas DataFrame for better data visualization and analysis
zcT = np.transpose(zc.get_zero_curve())
zc_dataframe = pd.DataFrame(zcT, columns=['Maturity', 'Discount Factor'])
zc_dataframe.set_index('Maturity', inplace=True)
print("\nZero Curve DataFrame:")
print(zc_dataframe)
zc_dataframe


2.5-year zero rate: 0.03100000000000001
1-year discount factor: 0.9851119396030628
2-year discount factor: 0.9512294245007139
3-year discount factor: 0.9003245225862656
4-year discount factor: 0.835270211411272
Zero rate for 1.5 years: 0.02166666666666669
Amount at Maturity for 1.5 years: 1.0330338931439726
Discount factor for 1.5 years: 0.968022449831306
Complete zero curve: ([0.5, 1, 2, 3, 4], [0.9937694906233947, 0.9851119396030628, 0.9512294245007139, 0.9003245225862656, 0.835270211411272])

Zero Curve DataFrame:
          Discount Factor
Maturity                 
0.5              0.993769
1.0              0.985112
2.0              0.951229
3.0              0.900325
4.0              0.835270


,Discount Factor
Maturity,
0.5,0.993769
1.0,0.985112
2.0,0.951229
3.0,0.900325
4.0,0.835270


In [37]:
class Bank_bill(CashFlows):  # Bank_bill inherits from CashFlows
    
    def __init__(self, face_value=100, maturity=0.25, ytm=0.00, price=100):
        # Call the parent class's __init__ method
        super().__init__()
        # Add attributes specific to bank bills
        self.face_value = face_value
        self.maturity = maturity
        self.ytm = ytm
        self.price = price
    
    def set_ytm(self, ytm):
        """Set yield to maturity and calculate the corresponding price."""
        self.ytm = ytm
        # Simple interest formula for bank bills
        self.price = self.face_value / (1 + ytm * self.maturity)
    
    def set_price(self, price):
        """Set price and calculate the implied yield to maturity."""
        self.price = price
        # Solve for ytm from the pricing formula
        self.ytm = (self.face_value / price - 1) / self.maturity
    
    def get_price(self):
        return self.price
    
    def get_ytm(self):
        return self.ytm
    
    def set_cash_flows(self):
        """Generate the cash flows for this bank bill."""
        self.add_cash_flow(0, -self.price)
        self.add_cash_flow(self.maturity, self.face_value)

# Create a 3-month bank bill with 2% yield
bill = Bank_bill(face_value=100, maturity=0.25, ytm=0.02)
bill.set_ytm(0.03)  # This calculates the price
bill.set_cash_flows()  # Generate the cash flows


print("Bank Bill:")
print(f"  Face Value: ${bill.face_value}")
print(f"  Maturity: {bill.maturity} years")
print(f"  Yield to Maturity: {bill.get_ytm():.2%}")
print(f"  Price: ${bill.get_price():.2f}")
print(f"  Cash Flows: {bill.get_cash_flows()}")

print(zc.npv(bill)) # net present value of the bank bill


Bank Bill:
  Face Value: $100
  Maturity: 0.25 years
  Yield to Maturity: 3.00%
  Price: $99.26
  Cash Flows: [(0, -99.25558312655086), (0.25, 100)]
-0.49815102444179615


In [39]:
class Bond(CashFlows):  # Bond inherits from CashFlows
    
    def __init__(self, face_value=100, maturity=3, coupon=0.05, frequency=4, ytm=0.05, price=100):
        # Call the parent class's __init__ method
        super().__init__()
        # Add attributes specific to bonds
        self.face_value = face_value
        self.maturity = maturity
        self.coupon = coupon
        self.frequency = frequency  # How many times per year coupons are paid
        self.ytm = ytm
        self.price = price
    
    def set_ytm(self, ytm):
        """Set yield to maturity and calculate the corresponding price using bond pricing formula."""
        self.ytm = ytm
        # Standard bond pricing formula
        coupon_payment = self.face_value * self.coupon / self.frequency
        n_periods = self.maturity * self.frequency
        rate_per_period = ytm / self.frequency
        
        # Present value of coupon payments (annuity)
        pv_coupons = coupon_payment * (1 - (1 + rate_per_period)**(-n_periods)) / rate_per_period
        # Present value of face value
        pv_face = self.face_value / ((1 + rate_per_period)**n_periods)
        
        self.price = pv_coupons + pv_face
    
    def get_price(self):
        return self.price
    
    def get_coupon_rate(self):
        return self.coupon
    
    def get_ytm(self):
        return self.ytm
    
    def set_cash_flows(self):
        """Generate all the cash flows for this bond."""
        # Initial outflow (purchase price)
        self.add_cash_flow(0, -self.price)
        
        # Coupon payments
        coupon_payment = self.face_value * self.coupon / self.frequency
        for i in range(1, self.maturity * self.frequency):
            self.add_cash_flow(i / self.frequency, coupon_payment)
        
        # Final payment (last coupon + face value)
        self.add_cash_flow(self.maturity, self.face_value + coupon_payment)

# Create a 2-year bond with 5% annual coupon, paid quarterly, yielding 4%
bond = Bond(face_value=100, maturity=2, coupon=0.05, frequency=4, ytm=0.04)
bond.set_ytm(0.04)  # This calculates the price
bond.set_cash_flows()  # Generate the cash flows

print("\nBond:")
print(f"  Face Value: ${bond.face_value}")
print(f"  Maturity: {bond.maturity} years")
print(f"  Coupon Rate: {bond.get_coupon_rate():.2%} (paid {bond.frequency} times per year)")
print(f"  Yield to Maturity: {bond.get_ytm():.2%}")
print(f"  Price: ${bond.get_price():.2f}")
print(f"\n  Cash Flows:")
for maturity, amount in bond.get_cash_flows():
    print(f"    Time {maturity:.2f}: ${amount:.2f}")


Bond:
  Face Value: $100
  Maturity: 2 years
  Coupon Rate: 5.00% (paid 4 times per year)
  Yield to Maturity: 4.00%
  Price: $101.91

  Cash Flows:
    Time 0.00: $-101.91
    Time 0.25: $1.25
    Time 0.50: $1.25
    Time 0.75: $1.25
    Time 1.00: $1.25
    Time 1.25: $1.25
    Time 1.50: $1.25
    Time 1.75: $1.25
    Time 2.00: $101.25


In [40]:
class Portfolio(CashFlows):
    
    def __init__(self):
        super().__init__()
        self.bonds = []
        self.bank_bills = []
    
    def add_bond(self, bond):
        """Add a bond to the portfolio."""
        self.bonds.append(bond)
        return f"Bond added (maturity: {bond.maturity} years)"
    
    def add_bank_bill(self, bank_bill):
        """Add a bank bill to the portfolio."""
        self.bank_bills.append(bank_bill)
        return f"Bank bill added (maturity: {bank_bill.maturity} years)"
    
    def get_bonds(self):
        return self.bonds
    
    def get_bank_bills(self):
        return self.bank_bills
    
    def set_cash_flows(self):
        """Aggregate all cash flows from all instruments in the portfolio."""
        # Add all cash flows from bonds
        for bond in self.bonds:
            for cash_flow in bond.get_cash_flows():
                self.add_cash_flow(cash_flow[0], cash_flow[1])
        
        # Add all cash flows from bank bills
        for bank_bill in self.bank_bills:
            for cash_flow in bank_bill.get_cash_flows():
                self.add_cash_flow(cash_flow[0], cash_flow[1])
    
    def total_value(self):
        """Calculate the total market value of the portfolio."""
        total = 0
        for bond in self.bonds:
            total += bond.get_price()
        for bill in self.bank_bills:
            total += bill.get_price()
        return total
    
    def list_instruments(self):
        """List all instruments in the portfolio."""
        if not self.bonds and not self.bank_bills:
            return "Portfolio is empty."
        
        result = ["Portfolio Contents:\n"]
        
        if self.bank_bills:
            result.append("Bank Bills:")
            for i, bill in enumerate(self.bank_bills, 1):
                result.append(f"  {i}. Maturity: {bill.maturity} years, "
                            f"Face Value: ${bill.face_value}, Price: ${bill.get_price():.2f}")
        
        if self.bonds:
            result.append("\nBonds:")
            for i, bond in enumerate(self.bonds, 1):
                result.append(f"  {i}. Maturity: {bond.maturity} years, "
                            f"Coupon: {bond.get_coupon_rate():.2%}, Price: ${bond.get_price():.2f}")
        
        result.append(f"\nTotal Portfolio Value: ${self.total_value():.2f}")
        
        return "\n".join(result)

# Create a portfolio
my_portfolio = Portfolio()

# Create some instruments
bill1 = Bank_bill(face_value=100, maturity=0.25, ytm=0.02)
bill1.set_ytm(0.02)
bill1.set_cash_flows()

bill2 = Bank_bill(face_value=100, maturity=0.5, ytm=0.025)
bill2.set_ytm(0.025)
bill2.set_cash_flows()

bond1 = Bond(face_value=100, maturity=1, coupon=0.04, frequency=2, ytm=0.03)
bond1.set_ytm(0.03)
bond1.set_cash_flows()

bond2 = Bond(face_value=100, maturity=2, coupon=0.05, frequency=4, ytm=0.04)
bond2.set_ytm(0.04)
bond2.set_cash_flows()

# Add instruments to portfolio
print(my_portfolio.add_bank_bill(bill1))
print(my_portfolio.add_bank_bill(bill2))
print(my_portfolio.add_bond(bond1))
print(my_portfolio.add_bond(bond2))

# List all instruments
print("\n" + my_portfolio.list_instruments())

# Generate aggregated cash flows for the entire portfolio
my_portfolio.set_cash_flows()

print("\nAggregated Portfolio Cash Flows:")
for maturity, amount in sorted(my_portfolio.get_cash_flows()):
    print(f"  Time {maturity:.2f}: ${amount:.2f}")

Bank bill added (maturity: 0.25 years)
Bank bill added (maturity: 0.5 years)
Bond added (maturity: 1 years)
Bond added (maturity: 2 years)

Portfolio Contents:

Bank Bills:
  1. Maturity: 0.25 years, Face Value: $100, Price: $99.50
  2. Maturity: 0.5 years, Face Value: $100, Price: $98.77

Bonds:
  1. Maturity: 1 years, Coupon: 4.00%, Price: $100.98
  2. Maturity: 2 years, Coupon: 5.00%, Price: $101.91

Total Portfolio Value: $401.16

Aggregated Portfolio Cash Flows:
  Time 0.00: $-101.91
  Time 0.00: $-100.98
  Time 0.00: $-99.50
  Time 0.00: $-98.77
  Time 0.25: $1.25
  Time 0.25: $100.00
  Time 0.50: $1.25
  Time 0.50: $2.00
  Time 0.50: $100.00
  Time 0.75: $1.25
  Time 1.00: $1.25
  Time 1.00: $102.00
  Time 1.25: $1.25
  Time 1.50: $1.25
  Time 1.75: $1.25
  Time 2.00: $101.25


In [43]:
import importlib
import curve_classes_and_functions as yCurve
importlib.reload(yCurve)
import instrument_classes as inst
importlib.reload(inst)
import pandas as pd
import math

# create a portfolio of two bank_bills
bank_bill_1 = inst.Bank_bill(face_value=100, maturity=.25)
bank_bill_1.set_ytm(.05)
bank_bill_1.set_cash_flows()
print(bank_bill_1.get_cash_flows())
bank_bill_2 = inst.Bank_bill(face_value=100, maturity=.5)
bank_bill_2.set_ytm(.06)
bank_bill_2.set_cash_flows()
print(bank_bill_2.get_cash_flows())
yc_portfolio = inst.Portfolio()
yc_portfolio.add_bank_bill(bank_bill_1)
yc_portfolio.add_bank_bill(bank_bill_2)
# print(yc_portfolio.get_cash_flows())

# create a yield curve based on the bank bill portfolio
yc=yCurve.YieldCurve()
yc.set_constituent_portfolio(yc_portfolio)
yc.bootstrap()
print(yc.get_zero_curve())

# create a bond and add it to the portfolio
bond = inst.Bond(face_value=100, maturity=1, coupon=.06, frequency=2)
bond.set_ytm(.07)
bond.set_cash_flows()
print(bond.get_cash_flows())
bond2 = inst.Bond(face_value=100, maturity=2, coupon=.08, frequency=1)
bond2.set_ytm(.09)
bond2.set_cash_flows()
print(bond2.get_cash_flows())

yc_portfolio.add_bond(bond)
yc_portfolio.add_bond(bond2)
yc_portfolio.set_cash_flows()
print(yc_portfolio.get_cash_flows())

yc2=yCurve.YieldCurve()
yc2.set_constituent_portfolio(yc_portfolio)
yc2.bootstrap()
print(yc2.get_zero_curve())

[(0, -98.76543209876543), (0.25, 100)]
[(0, -97.08737864077669), (0.5, 100)]
([0, 0.25, 0.5], [1.0, 0.9876543209876543, 0.9708737864077669])
[(0, -99.05015286237719), (0.5, 3.0), (1, 103.0)]
[(0, -98.24088881407287), (1.0, 8.0), (2, 108.0)]
[(0, -99.05015286237719), (0.5, 3.0), (1, 103.0), (0, -98.24088881407287), (1.0, 8.0), (2, 108.0), (0, -98.76543209876543), (0.25, 100), (0, -97.08737864077669), (0.5, 100)]
([0, 0.25, 0.5, 1, 2], [1.0, 0.9876543209876543, 0.9708737864077669, 0.93337409226363, 0.8404990377404059])
